# Selective DNN units
- Approach
    - Use functional localizer (fLOC) to identify face-, body-, and mixed-selective units in several DNNs
    - Confirm selectivity of units (d') using independent validation dataset

- Models
    - Main:
        - AlexNet-Sup. (Ecoset) + AlexNet-Self. (ImageNet)
    - Supplementary:
        - AlexNet-Sup. (ImageNet) + AlexNet-Self. (Ecoset) + AlexNet (untrained) + AlexNetGN (untrained)
        - VGG16-Sup. (Ecoset) + VGG16-Sup. (ImageNet) + VGG16 (untrained)

- fLoc dataset
    - images of faces, bodies, scenes, and objects (288 each), as well as scrambled images (144) (Stigliani et al., 2015)

- Validation dataset
    - Images of faces, bodies, and objects (150 each; Hemera Photo objects)


In [ ]:
from myutils.models import ModelLoader
from myutils.dataset import ImageDataLoader
from myutils.feature_extractor import FeatureExtractor
from facebody.config import DATA_ROOT
from facebody.selectivity import DNNfloc, compute_sel_dprime
from facebody.plotting import (
    plot_perc_layers, plot_perc_layers_mixed,
    plot_dprime_layers, plot_dprime_dist,
    plot_guided_gradcam,
)

device = "cuda:0"

ALEXNET_LAYERS = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
VGG16_LAYERS = [f"conv{i}" for i in range(1, 14)] + [f"fc{i}" for i in range(14, 16)]

## Models and layers

In [2]:
# AlexNet models
model_names = [
    "alexnet_ecoset", "alexnet_imagenet", # Supervised
    "alexnet-barlow-twins_ecoset", "alexnet-barlow-twins_imagenet", # Self-supervised
    "alexnet_untrained", "alexnet-barlow-twins_untrained", # Untrained
]
layers = ALEXNET_LAYERS

In [ ]:
# VGG16 models
model_names = ["vgg16_ecoset", "vgg16_imagenet"] # Supervised
layers = VGG16_LAYERS

## Identify selective units

In [ ]:
# Find selective units
floc_loader = ImageDataLoader(
    root=DATA_ROOT / "datasets" / "floc",
    class_names=["face", "body", "scene", "object", "scrambled"],
)

for model in model_names:
    ml = ModelLoader(model, DATA_ROOT, device)

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(floc_loader)

    dnn_floc = DNNfloc(
        model, layers,
        class_to_idx=floc_loader.dataset.class_to_idx,
        single_cats=["face", "body"],
        mixed_tuples=[("face", "body")],
    )
    dnn_floc.find_sel_units(activs, labels)

In [ ]:
# Fig. 1B: Plot percentage of selective units across layers
model_names = ["alexnet_ecoset", "alexnet-barlow-twins_imagenet"]
model_names_legend = ["AlexNet-Sup.\n(Ecoset)", "AlexNet-Self.\n(ImageNet)"]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_perc_layers(
    model_names, model_names_legend, layers,
    out_path="main/perc_layers.pdf",
)

In [ ]:
# Fig. S1: Plot percentage of selective units across layers
# Supervised models
model_names = ["alexnet_ecoset", "alexnet_imagenet"]
model_names_legend=["Ecoset", "ImageNet"]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_perc_layers(
    model_names, model_names_legend, layers,
    fig_title="AlexNet-Sup.", out_path="suppl/perc_layers_sup.pdf",
)

# Self-supervised models
model_names = ["alexnet-barlow-twins_ecoset", "alexnet-barlow-twins_imagenet"]
model_names_legend=["Ecoset", "ImageNet"]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_perc_layers(
    model_names, model_names_legend, layers,
    fig_title="AlexNet-Self.", out_path="suppl/perc_layers_self.pdf",
)

# Different architectures
model_names = ["vgg16_ecoset", "vgg16_imagenet"]
model_names_legend = ["Ecoset", "ImageNet"]
layers = [f"conv{i}" for i in range(1, 14)] + [f"fc{i}" for i in range(14, 16)]
plot_perc_layers(
    model_names, model_names_legend, layers,
    fig_title="VGG16-Sup.", out_path="suppl/perc_layers_vgg16.pdf",
)

# Untrained models
model_names = ["alexnet_untrained"]
model_names_legend = ["Untrained"]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_perc_layers(
    model_names, model_names_legend, layers,
    fig_title="AlexNet Untrained", out_path="suppl/perc_layers_alexnet_untrained.pdf",
)

model_names = ["alexnet-barlow-twins_untrained"]
model_names_legend = ["Untrained"]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_perc_layers(
    model_names, model_names_legend, layers,
    fig_title="AlexNet-GN Untrained", out_path="suppl/perc_layers_alexnet-barlow-twins_untrained.pdf",
)

model_names = ["vgg16_untrained"]
model_names_legend = ["Untrained"]
layers = [f"conv{i}" for i in range(1, 14)] + [f"fc{i}" for i in range(14, 16)]
plot_perc_layers(
    model_names, model_names_legend, layers,
    fig_title="VGG16 Untrained", out_path="suppl/perc_layers_vgg16_untrained.pdf",
)

In [ ]:
# Fig. S2: Plot percentage of mixed-selective units for other combinations across layers
model_names = ["alexnet_ecoset", "alexnet_imagenet"]
model_names_legend = ["Ecoset", "ImageNet"]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_perc_layers_mixed(
    model_names, model_names_legend, layers,
    fig_title="AlexNet-Sup.", out_path="suppl/mixed_perc_layers_sup.pdf",
)

model_names = ["alexnet-barlow-twins_ecoset", "alexnet-barlow-twins_imagenet"]
model_names_legend = ["Ecoset", "ImageNet"]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_perc_layers_mixed(
    model_names, model_names_legend, layers,
    fig_title="AlexNet-Self.", out_path="suppl/mixed_perc_layers_self.pdf",
)

model_names = ["vgg16_ecoset", "vgg16_imagenet"]
model_names_legend = ["Ecoset", "ImageNet"]
layers = [f"conv{i}" for i in range(1, 14)] + [f"fc{i}" for i in range(14, 16)]
plot_perc_layers_mixed(
    model_names, model_names_legend, layers, ylim=(0, 8),
    fig_title="VGG16-Sup.", out_path="suppl/mixed_perc_layers_vgg16.pdf",
)

In [ ]:
# Fig. S3A: Plot selectivity of units based on fLoc set across layers
model_names = [
    "alexnet_ecoset", "alexnet_imagenet",
    "alexnet-barlow-twins_ecoset", "alexnet-barlow-twins_imagenet",
]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_dprime_layers(
    model_names, layers, img_db="floc", fig_title="Localizer",
    subplot_titles=["AlexNet-Sup.", "AlexNet-Self."],
    out_path="suppl/dprime_layers_floc_alexnet.pdf",
)

model_names = [
    "alexnet_untrained", "alexnet-barlow-twins_untrained",
]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_dprime_layers(
    model_names, layers, img_db="floc", fig_title="Localizer",
    subplot_titles=["AlexNet Untrained", "AlexNet-GN. Untrained"],
    out_path="suppl/dprime_layers_floc_alexnet_untrained.pdf",
)

model_names = [
    "vgg16_ecoset", "vgg16_imagenet",
    "vgg16_untrained",
]
layers = [f"conv{i}" for i in range(1, 14)] + [f"fc{i}" for i in range(14, 16)]
plot_dprime_layers(
    model_names, layers, img_db="floc", fig_title="Localizer",
    subplot_titles=["VGG16-Sup.", "VGG16 Untrained"],
    out_path="suppl/dprime_layers_floc_vgg16.pdf",
)

## Validate selective units

In [ ]:
# Confirm selectivity (d')
vl = ImageDataLoader(root=DATA_ROOT / "datasets" / "validation")

for model in model_names:
    ml = ModelLoader(model, DATA_ROOT, device)

    fe = FeatureExtractor(ml, layers)
    activs, labels = fe.extract(vl)

    compute_sel_dprime(
        model, activs, labels, class_to_idx=vl.dataset.class_to_idx,
        img_db="validation", baseline_cat="object",
    )

In [ ]:
# Fig. 1C: Plot face and body selectivity aggregated across layers
model_names = ["alexnet_ecoset", "alexnet-barlow-twins_imagenet"]
model_names_legend = ["AlexNet-Sup.\n(Ecoset)", "AlexNet-Self.\n(ImageNet)"]
plot_dprime_dist(
    model_names, model_names_legend, ALEXNET_LAYERS,
    out_path="main/dprime_dist.pdf",
)

In [ ]:
# Fig. S3B: Plot selectivity of units based on validation set across layers
model_names = [
    "alexnet_ecoset", "alexnet_imagenet",
    "alexnet-barlow-twins_ecoset", "alexnet-barlow-twins_imagenet",
]
plot_dprime_layers(
    model_names, ALEXNET_LAYERS, img_db="validation", fig_title="Validation",
    subplot_titles=["AlexNet-Sup.", "AlexNet-Self."],
    out_path="suppl/dprime_layers_val_alexnet.pdf",
)

model_names = [
    "alexnet_untrained", "alexnet-barlow-twins_untrained",
]
layers = ["conv1", "conv2", "conv3", "conv4", "conv5", "fc6", "fc7"]
plot_dprime_layers(
    model_names, ALEXNET_LAYERS, img_db="validation", fig_title="Validation",
    subplot_titles=["AlexNet Untrained", "AlexNet-GN Untrained"],
    out_path="suppl/dprime_layers_val_alexnet_untrained.pdf",
)

model_names = [
    "vgg16_ecoset", "vgg16_imagenet",
    "vgg16_untrained",
]
layers = [f"conv{i}" for i in range(1, 14)] + [f"fc{i}" for i in range(14, 16)]
plot_dprime_layers(
    model_names, VGG16_LAYERS, img_db="validation", fig_title="Validation",
    subplot_titles=["VGG16-Sup.", "VGG16 Untrained"],
    out_path="suppl/dprime_layers_val_vgg16.pdf",
)

In [ ]:
# Fig. 1D: Plot saliency maps for unit types
layer = "conv5"
plot_guided_gradcam(
    model_name="alexnet_ecoset", target_layer=layer, cam_layer=None, controlled=True, device=device,
    img_path=DATA_ROOT / "datasets/guided_gradcam_examples_personal", n_imgs=2, norm_types=True,
    seed=0, fig_title=f"AlexNet-Sup. Ecoset\n{layer}", out_path="main/guided_gradcam.png",
)

In [ ]:
# Fig. S4: Plot saliency maps for unit types
layer = "conv5"
plot_guided_gradcam(
    model_name="alexnet_ecoset", target_layer=layer, cam_layer=None, controlled=True, device=device,
    img_path=DATA_ROOT / "datasets/guided_gradcam_examples", n_imgs=8, norm_types=True,
    seed=0, fig_title=f"AlexNet-Sup. Ecoset\n{layer}", out_path="suppl/guided_gradcam.pdf",
)